# Hotel Bookings Data Analysis

**Atliq Hotels** | May – July 2022 (with August sample data)

**Libraries:** Pandas, NumPy, Matplotlib, Seaborn

---

### Project Overview
End-to-end exploratory data analysis on hotel booking transactions and occupancy data to generate business insights on revenue, occupancy, cancellations, and key performance metrics (ADR, DBRN, DSRN, DURN).


## 1. Import Libraries


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print('Libraries imported successfully')


## 2. Load Datasets


In [ ]:
df_bookings = pd.read_csv('datasets/fact_bookings.csv')
df_agg_bookings = pd.read_csv('datasets/fact_aggregated_bookings.csv')
df_hotels = pd.read_csv('datasets/dim_hotels.csv')
df_rooms = pd.read_csv('datasets/dim_rooms.csv')
df_date = pd.read_csv('datasets/dim_date.csv')
df_august = pd.read_csv('datasets/new_data_august.csv')

print('fact_bookings      :', df_bookings.shape)
print('fact_aggregated    :', df_agg_bookings.shape)
print('dim_hotels         :', df_hotels.shape)
print('dim_rooms          :', df_rooms.shape)
print('dim_date           :', df_date.shape)
print('new_data_august    :', df_august.shape)


In [ ]:
df_bookings.head()


In [ ]:
df_bookings.info()


In [ ]:
print('Room categories :', df_bookings.room_category.unique())
print('Booking platforms:', df_bookings.booking_platform.unique())
print('\nBooking status distribution:')
print(df_bookings.booking_status.value_counts())


In [ ]:
df_bookings.booking_platform.value_counts().plot(kind='barh', color='steelblue')
plt.title('Bookings by Platform')
plt.xlabel('Number of Bookings')
plt.tight_layout()
plt.show()


In [ ]:
df_hotels.head()


In [ ]:
print(df_hotels.category.value_counts())
print()
print(df_hotels.city.value_counts())

df_hotels.city.value_counts().plot(kind='bar', color='teal')
plt.title('Number of Hotels by City')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 3. Explore Aggregate Bookings


In [ ]:
df_agg_bookings.head()


In [ ]:
print('Unique property IDs:', df_agg_bookings.property_id.nunique())
print(df_agg_bookings.property_id.unique())


In [ ]:
df_agg_bookings.groupby('property_id')['successful_bookings'].sum().sort_values(ascending=False)


In [ ]:
overbooked = df_agg_bookings[df_agg_bookings.successful_bookings > df_agg_bookings.capacity]
print('Records with successful_bookings > capacity:', len(overbooked))
overbooked.head()


In [ ]:
print('Max capacity:', df_agg_bookings.capacity.max())
print('Property with highest capacity:')
print(df_agg_bookings.loc[df_agg_bookings.capacity.idxmax()])


## 4. Data Cleaning


### 4.1 Clean invalid guest counts


In [ ]:
print('Rows with no_guests <= 0:', (df_bookings.no_guests <= 0).sum())
df_bookings = df_bookings[df_bookings.no_guests > 0].copy()
print('Shape after removing invalid guests:', df_bookings.shape)


### 4.2 Outlier removal in revenue_generated


In [ ]:
print('Revenue generated – min / max / mean / median:')
print(df_bookings.revenue_generated.min(),
      df_bookings.revenue_generated.max(),
      round(df_bookings.revenue_generated.mean(), 2),
      df_bookings.revenue_generated.median())


In [ ]:
avg = df_bookings.revenue_generated.mean()
std = df_bookings.revenue_generated.std()
higher_limit = avg + 3 * std

print(f'Mean + 3*std upper limit: {higher_limit:.0f}')
print(f'Outliers above limit: {(df_bookings.revenue_generated > higher_limit).sum()}')

df_bookings = df_bookings[df_bookings.revenue_generated <= higher_limit].copy()
print('Shape after outlier removal:', df_bookings.shape)


In [ ]:
df_bookings.revenue_realized.describe()


### 4.3 Handle missing capacity in aggregate bookings


In [ ]:
print('Missing capacity values:', df_agg_bookings.capacity.isna().sum())
df_agg_bookings['capacity'] = df_agg_bookings['capacity'].fillna(df_agg_bookings.capacity.median())
print('Missing capacity after fill:', df_agg_bookings.capacity.isna().sum())


## 5. Data Transformation


### 5.1 Create occupancy percentage column


In [ ]:
df_agg_bookings['occ_pct'] = (df_agg_bookings['successful_bookings'] / df_agg_bookings['capacity'] * 100).round(2)
df_agg_bookings.head()


## 6. Insights Generation


### 6.1 Average occupancy rate by room category


In [ ]:
occ_by_room = df_agg_bookings.groupby('room_category')['occ_pct'].mean().round(2)
print(occ_by_room)

occ_room = df_agg_bookings.merge(df_rooms, left_on='room_category', right_on='room_id', how='left')
occ_by_class = occ_room.groupby('room_class')['occ_pct'].mean().reindex(
    ['Standard', 'Elite', 'Premium', 'Presidential']
).round(2)
print('\nBy room class:')
print(occ_by_class)

occ_by_class.plot(kind='bar', color='steelblue')
plt.title('Average Occupancy % by Room Class')
plt.ylabel('Occupancy %')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### 6.2 Occupancy rate by city


In [ ]:
df_agg_city = df_agg_bookings.merge(df_hotels, on='property_id', how='left')
occ_by_city = df_agg_city.groupby('city')['occ_pct'].mean().sort_values(ascending=False).round(2)
print(occ_by_city)

occ_by_city.plot(kind='bar', color='teal')
plt.title('Average Occupancy % by City')
plt.ylabel('Occupancy %')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### 6.3 Weekday vs Weekend occupancy


In [ ]:
df_agg_bookings['check_in_date'] = pd.to_datetime(df_agg_bookings['check_in_date'], dayfirst=True, errors='coerce')
df_date['date'] = pd.to_datetime(df_date['date'], dayfirst=True, errors='coerce')

df_agg_date = df_agg_bookings.merge(df_date, left_on='check_in_date', right_on='date', how='left')
occ_day_type = df_agg_date.groupby('day_type')['occ_pct'].mean().round(2)
print(occ_day_type)

occ_day_type.plot(kind='bar', color=['#3498db', '#e74c3c'])
plt.title('Average Occupancy %: Weekday vs Weekend')
plt.ylabel('Occupancy %')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### 6.4 June occupancy by city


In [ ]:
june_data = df_agg_date[df_agg_date['mmm yy'] == 'Jun 22']
june_city = june_data.merge(df_hotels, on='property_id', how='left')
june_occ = june_city.groupby('city')['occ_pct'].mean().sort_values(ascending=False).round(2)
print('June 2022 Occupancy by City:')
print(june_occ)

june_occ.plot(kind='bar', color='coral')
plt.title('June 2022 – Occupancy % by City')
plt.ylabel('Occupancy %')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### 6.5 Append August data


In [ ]:
print('August data sample:')
print(df_august.head())
print('\nAugust shape:', df_august.shape)
print('Columns:', df_august.columns.tolist())


In [ ]:
print('August occupancy summary:')
print(df_august[['property_name', 'city', 'successful_bookings', 'capacity', 'occ%']].head(10))


### 6.6 Revenue realized per city


In [ ]:
df_bookings_hotels = df_bookings.merge(df_hotels, on='property_id', how='left')

rev_city = df_bookings_hotels.groupby('city')['revenue_realized'].sum().sort_values(ascending=False)
print(rev_city)

rev_city.plot(kind='bar', color='green')
plt.title('Total Revenue Realized by City')
plt.ylabel('Revenue (₹)')
plt.xticks(rotation=0)
plt.ticklabel_format(style='plain', axis='y')
plt.tight_layout()
plt.show()


### 6.7 Month-by-month revenue


In [ ]:
df_bookings['check_in_date'] = pd.to_datetime(df_bookings['check_in_date'], dayfirst=True, errors='coerce')
df_bookings_date = df_bookings.merge(df_date, left_on='check_in_date', right_on='date', how='left')

rev_month = df_bookings_date.groupby('mmm yy')['revenue_realized'].sum()
month_order = ['May 22', 'Jun 22', 'Jul 22']
rev_month = rev_month.reindex([m for m in month_order if m in rev_month.index])
print(rev_month)

rev_month.plot(kind='bar', color='purple')
plt.title('Month-by-Month Revenue Realized')
plt.ylabel('Revenue (₹)')
plt.xticks(rotation=0)
plt.ticklabel_format(style='plain', axis='y')
plt.tight_layout()
plt.show()


## 7. Revenue Analysis


### 7.1 Hotel-wise revenue


In [ ]:
hotel_revenue = (
    df_bookings_hotels
    .groupby(['property_name', 'city'])
    .agg(total_revenue=('revenue_realized', 'sum'),
         bookings=('booking_id', 'count'))
    .sort_values('total_revenue', ascending=False)
    .reset_index()
)
hotel_revenue.head(10)


In [ ]:
top10 = hotel_revenue.head(10)
plt.figure(figsize=(11, 6))
sns.barplot(data=top10, y='property_name', x='total_revenue', hue='city', dodge=False, palette='tab10')
plt.title('Top 10 Hotels by Revenue Realized')
plt.xlabel('Revenue Realized (₹)')
plt.ylabel('')
plt.legend(title='City', loc='lower right')
plt.tight_layout()
plt.show()


### 7.2 City-wise revenue summary


In [ ]:
city_summary = (
    df_bookings_hotels
    .groupby('city')
    .agg(total_revenue=('revenue_realized', 'sum'),
         bookings=('booking_id', 'count'),
         avg_rating=('ratings_given', 'mean'))
    .round(2)
    .sort_values('total_revenue', ascending=False)
)
city_summary


### 7.3 Revenue per week


In [ ]:
rev_week = (
    df_bookings_date
    .groupby('week no')['revenue_realized']
    .sum()
    .reset_index()
)
rev_week.head(10)


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(rev_week['week no'], rev_week['revenue_realized'], marker='o', color='darkblue')
plt.title('Weekly Revenue Trend')
plt.xlabel('Week Number')
plt.ylabel('Revenue Realized (₹)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 8. Booking Analysis


### 8.1 Hotel-wise cancellation / checkout / no-show percentage


In [ ]:
status_hotel = (
    df_bookings_hotels
    .groupby(['property_name', 'booking_status'])
    .size()
    .unstack(fill_value=0)
)
status_hotel['total'] = status_hotel.sum(axis=1)
status_hotel['cancel_pct'] = (status_hotel.get('Cancelled', 0) / status_hotel['total'] * 100).round(2)
status_hotel['checkout_pct'] = (status_hotel.get('Checked Out', 0) / status_hotel['total'] * 100).round(2)
status_hotel['noshow_pct'] = (status_hotel.get('No Show', 0) / status_hotel['total'] * 100).round(2)

status_hotel[['cancel_pct', 'checkout_pct', 'noshow_pct']].sort_values('cancel_pct', ascending=False).head(10)


### 8.2 Platform-wise bookings


In [ ]:
platform_stats = (
    df_bookings
    .groupby('booking_platform')
    .agg(bookings=('booking_id', 'count'),
         revenue=('revenue_realized', 'sum'),
         avg_rating=('ratings_given', 'mean'))
    .sort_values('bookings', ascending=False)
    .round(2)
)
platform_stats


In [ ]:
platform_stats['bookings'].plot(kind='barh', color='steelblue')
plt.title('Bookings by Platform')
plt.xlabel('Number of Bookings')
plt.tight_layout()
plt.show()


### 8.3 Room category-wise bookings


In [ ]:
room_bookings = (
    df_bookings
    .merge(df_rooms, left_on='room_category', right_on='room_id', how='left')
    .groupby('room_class')
    .agg(bookings=('booking_id', 'count'),
         revenue=('revenue_realized', 'sum'))
    .reindex(['Standard', 'Elite', 'Premium', 'Presidential'])
)
room_bookings


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
room_bookings['bookings'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Bookings by Room Class')
axes[0].set_ylabel('Bookings')
axes[0].tick_params(axis='x', rotation=0)

room_bookings['revenue'].plot(kind='bar', ax=axes[1], color='green')
axes[1].set_title('Revenue by Room Class')
axes[1].set_ylabel('Revenue (₹)')
axes[1].tick_params(axis='x', rotation=0)
axes[1].ticklabel_format(style='plain', axis='y')

plt.tight_layout()
plt.show()


## 9. Occupancy Analysis


### 9.1 Hotel-wise occupancy


In [ ]:
hotel_occ = (
    df_agg_city
    .groupby(['city', 'property_name'])
    .agg(successful_bookings=('successful_bookings', 'sum'),
         capacity=('capacity', 'sum'))
    .reset_index()
)
hotel_occ['occupancy'] = (hotel_occ['successful_bookings'] * 100 / hotel_occ['capacity']).round(2)
hotel_occ.sort_values('occupancy', ascending=False).head(10)


### 9.2 Room class-wise occupancy


In [ ]:
room_occ = (
    occ_room
    .groupby('room_class')
    .agg(successful_bookings=('successful_bookings', 'sum'),
         capacity=('capacity', 'sum'))
)
room_occ['occupancy'] = (room_occ['successful_bookings'] * 100 / room_occ['capacity']).round(2)
room_occ = room_occ.reindex(['Standard', 'Elite', 'Premium', 'Presidential'])
room_occ


### 9.3 Day-wise occupancy


In [ ]:
df_agg_date['Day'] = df_agg_date['check_in_date'].dt.day_name()
df_agg_date['Day_No'] = df_agg_date['check_in_date'].dt.dayofweek

day_occ = (
    df_agg_date
    .groupby(['Day', 'Day_No'])
    .agg(successful_bookings=('successful_bookings', 'sum'),
         capacity=('capacity', 'sum'))
    .reset_index()
)
day_occ['occupancy'] = (day_occ['successful_bookings'] * 100 / day_occ['capacity']).round(2)
day_occ = day_occ.sort_values('Day_No')
day_occ[['Day', 'occupancy']]


In [ ]:
plt.figure(figsize=(9, 4))
sns.barplot(data=day_occ, x='Day', y='occupancy', order=day_occ['Day'].tolist(), palette='Blues_d')
plt.title('Average Occupancy % by Day of Week')
plt.ylabel('Occupancy %')
plt.xlabel('')
plt.tight_layout()
plt.show()


### 9.4 Week-wise occupancy


In [ ]:
weekly_occ = (
    df_agg_date
    .groupby('week no')
    .agg(successful_bookings=('successful_bookings', 'sum'),
         capacity=('capacity', 'sum'))
    .reset_index()
)
weekly_occ['occupancy'] = (weekly_occ['successful_bookings'] * 100 / weekly_occ['capacity']).round(2)
weekly_occ.head(10)


## 10. Advanced Metrics – ADR, DBRN, DSRN, DURN

### 10.1 ADR – Average Daily Rate

ADR = Total Revenue Realized / Number of Bookings


In [ ]:
adr_df = (
    df_bookings_hotels
    .groupby('property_name')
    .agg(bookings=('booking_id', 'count'),
         revenue_realized=('revenue_realized', 'sum'))
    .reset_index()
)
adr_df['ADR'] = (adr_df['revenue_realized'] / adr_df['bookings']).round(2)
adr_df = adr_df.sort_values('ADR', ascending=False)
adr_df[['property_name', 'ADR']].head(10)


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=adr_df.head(10), y='property_name', x='ADR', palette='viridis')
plt.title('Top 10 Hotels by ADR (Average Daily Rate)')
plt.xlabel('ADR (₹)')
plt.ylabel('')
plt.tight_layout()
plt.show()


### 10.2 DBRN – Day Booking Revenue per Night

DBRN = Successful (Checked Out) bookings / Total days in period


In [ ]:
total_days = df_date['date'].nunique()
print('Total days in period:', total_days)

dbrn_df = (
    df_bookings_hotels[df_bookings_hotels['booking_status'] == 'Checked Out']
    .groupby('property_name')
    .agg(successful_bookings=('booking_id', 'count'))
    .reset_index()
)
dbrn_df['DBRN'] = (dbrn_df['successful_bookings'] / total_days).round(2)
dbrn_df = dbrn_df.sort_values('DBRN', ascending=False)
dbrn_df[['property_name', 'DBRN']].head(10)


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=dbrn_df.head(10), y='property_name', x='DBRN', palette='Blues_r')
plt.title('Top 10 Hotels by DBRN')
plt.xlabel('DBRN')
plt.ylabel('')
plt.tight_layout()
plt.show()


### 10.3 DSRN – Day Sales (Capacity) per Night

DSRN = Total capacity / Total days


In [ ]:
hotel_capacity = (
    df_agg_city
    .groupby('property_name')
    .agg(total_capacity=('capacity', 'sum'))
    .reset_index()
)
hotel_capacity['DSRN'] = (hotel_capacity['total_capacity'] / total_days).round(2)
hotel_capacity = hotel_capacity.sort_values('DSRN', ascending=False)
hotel_capacity[['property_name', 'DSRN']].head(10)


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=hotel_capacity.head(10), y='property_name', x='DSRN', palette='Greens_r')
plt.title('Top 10 Hotels by DSRN')
plt.xlabel('DSRN')
plt.ylabel('')
plt.tight_layout()
plt.show()


### 10.4 DURN – Day Utilization Rate per Night

DURN = Checked Out bookings / Total days


In [ ]:
durn_df = (
    df_bookings_hotels[df_bookings_hotels['booking_status'] == 'Checked Out']
    .groupby('property_name')
    .agg(checked_out=('booking_id', 'count'))
    .reset_index()
)
durn_df['DURN'] = (durn_df['checked_out'] / total_days).round(2)
durn_df = durn_df.sort_values('DURN', ascending=False)
durn_df[['property_name', 'DURN']].head(10)


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=durn_df.head(10), y='property_name', x='DURN', palette='Oranges_r')
plt.title('Top 10 Hotels by DURN')
plt.xlabel('DURN')
plt.ylabel('')
plt.tight_layout()
plt.show()


## 11. Summary of Key Insights

1. **Data Quality**
   - Removed rows with `no_guests <= 0`.
   - Removed extreme `revenue_generated` outliers (mean + 3×std).
   - Filled missing capacity values with median.

2. **Occupancy**
   - Occupancy varies by room class and city.
   - Weekend vs weekday differences are visible.
   - June occupancy by city provides a monthly snapshot.

3. **Revenue**
   - Clear leaders among cities and individual hotels.
   - Monthly and weekly trends show seasonality.

4. **Bookings & Cancellations**
   - Platform mix and cancellation rates differ across hotels and channels.
   - Standard and Elite rooms drive volume; higher classes drive revenue per booking.

5. **Advanced Metrics**
   - **ADR**, **DBRN**, **DSRN**, and **DURN** help compare hotels on rate, booking intensity, capacity, and utilization.

---
*Analysis completed with Pandas, NumPy, Matplotlib & Seaborn.*
